# PDEs in Finance and Beyond

The Black-Scholes equation for an option price $V(S, t)$ is a heat equation in disguise. This notebook solves it by the method of lines, checks the result against the Black-Scholes formula, and reads off the hedge ratio. It then solves the heat equation a second way, writing the PDE symbolically and letting **MethodOfLines.jl** build the grid.

Install once with `] add Distributions ModelingToolkit MethodOfLines DomainSets` if needed.

In [ ]:
using OrdinaryDiffEq, SparseArrays, LinearAlgebra, Distributions, Plots

## The Black-Scholes equation

With time to maturity $\tau = T - t$, a European call satisfies

$$V_\tau = \tfrac12\sigma^2S^2V_{SS} + rSV_S - rV, \qquad V(S, 0) = \max(S - K, 0),$$

with $V(0, \tau) = 0$ and $V(S, \tau) \approx S - Ke^{-r\tau}$ for large $S$. Central differences on a grid in $S$ give a tridiagonal system, which is solved forward in $\tau$.

In [ ]:
K, r, sigma, T = 105.0, 0.03, 0.2, 1.0
Smax, M = 400.0, 800
hS = Smax / M; S = hS .* (1:M-1)                     # interior grid
a = 0.5 .* sigma^2 .* S .^ 2 ./ hS^2; b = r .* S ./ (2hS)
Aop = spdiagm(-1 => (a .- b)[2:end], 0 => -2 .* a .- r, 1 => (a .+ b)[1:end-1])
edge(tau) = [zeros(M - 2); (a[end] + b[end]) * (Smax - K * exp(-r * tau))]   # the known value at S = Smax
bs(V, p, tau) = Aop * V .+ edge(tau)
sol = solve(ODEProblem(bs, max.(S .- K, 0.0), (0.0, T)), FBDF(); reltol=1e-8, abstol=1e-10)

In [ ]:
function bs_formula(S0)
    d1 = (log(S0 / K) + (r + sigma^2 / 2) * T) / (sigma * sqrt(T)); d2 = d1 - sigma * sqrt(T)
    return S0 * cdf(Normal(), d1) - K * exp(-r * T) * cdf(Normal(), d2), cdf(Normal(), d1)
end
V = sol.u[end]
i = findfirst(==(100.0), S)
(finite_difference_price = V[i], formula_price = bs_formula(100.0)[1],
 finite_difference_delta = (V[i + 1] - V[i - 1]) / (2hS), formula_delta = bs_formula(100.0)[2])

In [ ]:
near = 50 .<= S .<= 160
plot(S[near], [max.(S[near] .- K, 0.0) V[near] first.(bs_formula.(S[near]))], lw=[2 3 1], ls=[:dot :solid :dash],
     label=["payoff at maturity" "finite differences, one year out" "Black-Scholes formula"], xlabel="stock price", ylabel="call value", legend=:topleft)

The hedge ratio $\Delta = V_S$ and the gamma $V_{SS}$ come straight from the grid, with no extra solves. Gamma is largest near the strike, where the payoff's kink has been smoothed by one year of "diffusion".

In [ ]:
delta = (V[3:end] .- V[1:end-2]) ./ (2hS)
gamma = (V[3:end] .- 2 .* V[2:end-1] .+ V[1:end-2]) ./ hS^2
mid = S[2:end-1]; w = 50 .<= mid .<= 160
plot(plot(mid[w], delta[w], lw=2, label="", title="Delta"), plot(mid[w], gamma[w], lw=2, label="", title="Gamma"), layout=(1, 2), size=(900, 320), xlabel="stock price")

## The heat equation, written symbolically

MethodOfLines.jl takes the PDE, the boundary conditions and the domains as symbolic equations (ModelingToolkit), builds the finite-difference system, and returns an ordinary `ODEProblem`. The result should match the hand-built grid of notebook 1 to solver tolerance.

In [ ]:
using ModelingToolkit, MethodOfLines, DomainSets
@parameters t x
@variables u(..)
Dt = Differential(t); Dxx = Differential(x)^2
eq = Dt(u(t, x)) ~ 0.01 * Dxx(u(t, x))
bcs = [u(0, x) ~ exp(-100 * (x - 0.5)^2), u(t, 0) ~ 0.0, u(t, 1) ~ 0.0]
domains = [t in Interval(0.0, 5.0), x in Interval(0.0, 1.0)]
@named pdesys = PDESystem(eq, bcs, domains, [t, x], [u(t, x)])
prob = discretize(pdesys, MOLFiniteDifference([x => 0.01], t))
msol = solve(prob, FBDF(); reltol=1e-10, abstol=1e-12)
xgrid = msol[x]; U = msol[u(t, x)]   # rows are times, columns are grid points
heatmap(xgrid, msol[t], U, c=:viridis, xlabel="x", ylabel="t", title="Heat equation via MethodOfLines.jl")

In [ ]:
n = 99; hx = 1 / (n + 1); xi = hx .* (1:n)
A = spdiagm(-1 => ones(n - 1), 0 => fill(-2.0, n), 1 => ones(n - 1)) ./ hx^2
hand = solve(ODEProblem((v, p, s) -> 0.01 .* (A * v), exp.(-100 .* (xi .- 0.5) .^ 2), (0.0, 5.0)), FBDF(); reltol=1e-10, abstol=1e-12)
(same_grid = xgrid[2:end-1] ≈ xi, largest_difference_at_t_5 = maximum(abs.(U[end, 2:end-1] .- hand.u[end])))

## Beyond

- **Turing patterns** (Turing, 1952): two reacting chemicals diffusing at different rates can create spots and stripes from a uniform state. This is the origin of animal coat patterns.
- **Mean-field games and heterogeneous-agent macroeconomics** couple a Hamilton-Jacobi-Bellman equation, solved backwards, with a Kolmogorov forward equation for the distribution of agents.
- **Finite elements** (Gridap.jl, Ferrite.jl) handle curved domains such as a lake, a lung or an engine part.

## Exercises

1. Price the put with the same strike: payoff $\max(K - S, 0)$, $V(0, \tau) = Ke^{-r\tau}$ and $V(S_{\max}, \tau) = 0$. Check put-call parity.
2. Halve the grid spacing in $S$. By how much does the error at $S = 100$ fall?
3. Add an early-exercise constraint for an American put by solving in small steps of $\tau$ and taking `max.(V, K .- S)` after each one. How large is the early-exercise premium?
4. In the MethodOfLines example, replace the right boundary by no flux, `Differential(x)(u(t, 1)) ~ 0`, and check that $\int u\,dx$ decays more slowly.